# Chapter 12 · Preparing Features for Machine Learning (in class)

**Goal for today:** turn a raw table of 20,057 recipes and 680 columns into
model-ready features, to predict whether a dish is a **dessert**.

Most of a machine learning project happens *before* the model. We will work through
ten general rules, one step at a time, and check the data after every step.

| # | Rule | Where we use it |
|---|------|-----------------|
| 1 | Start from the question: know the target and what one row is | Step 1 |
| 2 | Fix the structure first: names and types before statistics | Steps 2 to 5 |
| 3 | Treat every missing value as a decision | Step 7 |
| 4 | Tame extreme values instead of deleting rows | Step 8 |
| 5 | Protect the target: no missing targets, no information from after the outcome | Step 7 |
| 6 | Drop features that cannot discriminate | Step 9 |
| 7 | Create features from domain knowledge | Step 10 |
| 8 | Remove redundancy (highly correlated features) | Step 11 |
| 9 | Put features on comparable scales | Step 14 |
| 10 | Make every step repeatable: learn from training data, apply to new data | Steps 12 to 15 |

### How to use this notebook

This is the in-class version. The explanations and the working code are here; the
piece each step is teaching has been replaced by `___`.

- Change `account` in the setup cell to your own Bryant username before you run anything.
- Fill in every `___` and run the cell. If it errors, read the message first: it
  usually names the column or function it could not find.
- Read each **Question to consider** before you run the step, and be ready to answer it.
- Run the **Check** cells: they show what each step changed.
- The **Try it** exercises have an empty cell: write that code yourself.

## Step 0 · Start Spark

In [1]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql.utils import AnalysisException
import pyspark.sql.functions as F
import pyspark.sql.types as T
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# change the account name to your email account
account='sli'

# define a root path to access the data in the DataAnalysisWithPythonAndPySpark
data_path='/net/clusterhn/home/'+account+'/isa460/data/'

# check if the Spark session is active. If it is activate, close it

try:
    if spark:
        spark.stop()
except:
    pass    

spark = (SparkSession.builder.appName("Preparing Featurers for Machine Learning")
        .config("spark.port.maxRetries", "100")
        .config("spark.sql.mapKeyDedupPolicy", "LAST_WIN")  # This configuration allow the duplicate keys in the map data type.
        .config("spark.driver.memory", "8g")
        .getOrCreate())

# confiture the log level (defaulty is WARN)
spark.sparkContext.setLogLevel('ERROR')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/18 16:34:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/18 16:34:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/18 16:34:56 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/10/18 16:34:56 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


---
## Step 1 · Load the data and name the question
**Rule 1.** Before touching anything, know what one row is and what we are predicting.

> **Question to consider:** One row is one recipe. What would go wrong if one row were one *ingredient* instead?

In [2]:
food = spark.read.csv(data_path + 'recipes/epi_r.csv', inferSchema=True, header=True)
print(food.count(), 'rows,', len(food.columns), 'columns')

20057 rows, 680 columns


In [3]:
food.select('title', 'rating', 'calories', 'protein', 'fat', 'sodium',
            '#cakeweek', '#wasteless', 'dessert').show(5)

+--------------------+------+--------+-------+----+------+---------+----------+-------+
|               title|rating|calories|protein| fat|sodium|#cakeweek|#wasteless|dessert|
+--------------------+------+--------+-------+----+------+---------+----------+-------+
|Lentil, Apple, an...|   2.5|   426.0|   30.0| 7.0| 559.0|      0.0|       0.0|    0.0|
|Boudin Blanc Terr...| 4.375|   403.0|   18.0|23.0|1439.0|      0.0|       0.0|    0.0|
|Potato and Fennel...|  3.75|   165.0|    6.0| 7.0| 165.0|      0.0|       0.0|    0.0|
|Mahi-Mahi in Toma...|   5.0|    NULL|   NULL|NULL|  NULL|      0.0|       0.0|    0.0|
|Spinach Noodle Ca...| 3.125|   547.0|   20.0|32.0| 452.0|      0.0|       0.0|    0.0|
+--------------------+------+--------+-------+----+------+---------+----------+-------+
only showing top 5 rows



In [5]:
# The target: how many desserts?
food.groupBy('dessert').count().show()

+-------+-----+
|dessert|count|
+-------+-----+
|    0.0|16479|
|   NULL|    5|
|    1.0| 3573|
+-------+-----+



**Check.** About 3,573 desserts against 16,479 other dishes, plus 5 recipes with no
target: roughly 18% positive. A model that always says "not a dessert" would be
82% accurate and useless, so accuracy alone will not be enough to judge it.

---
## Step 2 · Standardize the column names
**Rule 2.** Names like `#cakeweek` or `22-minute meals` break dot notation and SQL.

> **Question to consider:** Why fix the names with one function instead of renaming columns one at a time?

In [ ]:
# Two blanks: the string method that makes a name lower case, and the
# DataFrame method that replaces ALL column names at once, in order.
def sanitize_column_name(name):
    # lowercase; spaces, dashes and slashes become _; & becomes and; drop the rest
    answer = name.___()
    for i, j in ((' ', '_'), ('-', '_'), ('/', '_'), ('&', 'and')):
        answer = answer.replace(i, j)
    return ''.join(c for c in answer if c.isalpha() or c.isdigit() or c == '_')


# toDF() takes the new names in order, so all 680 columns are renamed at once
food1 = food.___(*[sanitize_column_name(name) for name in food.columns])

In [ ]:
# Check: which names changed?
changed = [(old, new) for old, new in zip(food.columns, food1.columns) if old != new]
print(len(changed), 'names changed, for example:')
changed[:8]

---
## Step 3 · Explore before you clean
**Rule 2.** Two quick checks reveal most problems: the spread of each number, and which columns are really flags.

> **Question to consider:** Protein has a median of 8 grams. What do you make of the maximum?

In [ ]:
food1.select('rating', 'calories', 'protein', 'fat', 'sodium').summary().show()

In [ ]:
# Which of the first 20 columns are binary (exactly two distinct values)?
is_binary = food1.agg(
    *[(F.countDistinct(c) == 2).alias(c) for c in food1.columns[0:20]]
).toPandas()
is_binary.T.rename(columns={0: 'is binary'})

---
## Step 4 · Fix data mishaps
**Rule 2.** A 0/1 flag that holds 880 is a broken record, not a recipe feature.

> **Question to consider:** We could replace 880 with 1. Why is dropping the row the safer choice?

In [ ]:
food1.groupBy('cakeweek').count().show()
food1.groupBy('wasteless').count().show()

In [ ]:
# Keep a row only if each flag is 0, 1 or missing. One blank checks membership
# in a list of values, the other checks for a missing value.
# keep only 0, 1 or missing in both flags
food2 = food1.where(
    (F.col('cakeweek').___([0.0, 1.0]) | F.col('cakeweek').isNull())
    & (F.col('wasteless').isin([0.0, 1.0]) | F.col('wasteless').___())
)
print(food1.count(), '->', food2.count())

---
## Step 5 · Correct the data types
**Rule 2.** Numbers stored as text cannot be summarized, compared or fed to a model.

> **Question to consider:** Why might Spark have read `rating` and `calories` as strings?

In [ ]:
string_columns = [c for c in food2.columns if food2.schema[c].dataType == T.StringType()]
string_columns

In [ ]:
# The Column method that converts a column to another data type.
food2 = (food2
         .withColumn('rating', F.col('rating').___('double'))
         .withColumn('calories', F.col('calories').cast('double')))
[c for c in food2.columns if food2.schema[c].dataType == T.StringType()]

---
## Step 6 · Name the column groups once
Each kind of column needs different treatment, so keep four lists and let every
later step use them.

In [ ]:
IDENTIFIERS = ['title']
CONTINUOUS_COLUMNS = ['rating', 'calories', 'protein', 'fat', 'sodium']
TARGET_COLUMN = ['dessert']
BINARY_COLUMNS = [
    c for c in food2.columns
    if c not in CONTINUOUS_COLUMNS and c not in TARGET_COLUMN and c not in IDENTIFIERS
]
print(len(BINARY_COLUMNS), 'binary columns')

---
## Step 7 · Missing values: drop, then fill
**Rule 3 and 5.** Three different missing-value problems get three different answers.

> **Question to consider:** A recipe's flag for 'anise' is missing. Is 0 a reasonable guess? Would 0 be reasonable for calories?

In [ ]:
# 1. drop a row only when ALL of its columns are missing;
# 2. drop rows with no target; 3. fill missing flags with 0.
# 1. rows where every column except the title is missing
food3 = food2.dropna(how='___', subset=[c for c in food2.columns if c not in IDENTIFIERS])
print('after dropping empty rows:  ', food3.count())

# 2. rows with no target: nothing to learn from (rule 5)
food3 = food3.___(subset=___)
print('after dropping missing target:', food3.count())

# 3. a missing flag most likely means the recipe does not have it
food3 = food3.___(0.0, subset=___)

In [ ]:
# Check: missing values left in the continuous columns (the Imputer handles these in Step 13)
food3.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in CONTINUOUS_COLUMNS]).show()

---
## Step 8 · Cap extreme values
**Rule 4.** Keep the row, limit the damage: anything above the 99th percentile is set to the 99th percentile.

> **Question to consider:** Why cap instead of deleting every recipe with an extreme value? And what should happen to a recipe with no calorie value at all?

In [ ]:
# The 99th percentile, written as a fraction.
selected_columns = ['calories', 'protein', 'fat', 'sodium']
cap = {c: food3.select(F.percentile_approx(c, ___)).first()[0] for c in selected_columns}
cap

In [ ]:
# The function that returns the smaller of two columns. Careful: it SKIPS
# nulls (it would turn a missing value into the cap), which is why the
# when() guard is there.
before = food3.select(selected_columns).toPandas()

for c, v in cap.items():
    # keep the smaller of the value and the cap; the when() keeps missing values missing
    food3 = food3.withColumn(c, F.when(F.col(c).isNull(), F.col(c))
                                 .otherwise(F.___(F.col(c), F.lit(v))))

after = food3.select(selected_columns).toPandas()

# Check: the missing values must still be missing (the Imputer fills them in Step 13)
pd.DataFrame({'missing before': before.isna().sum(), 'missing after': after.isna().sum()})

In [ ]:
fig, axes = plt.subplots(len(selected_columns), 2, figsize=(11, 10))
for i, c in enumerate(selected_columns):
    sns.boxplot(x=before[c], ax=axes[i, 0]).set_title(f'{c}: before')
    sns.boxplot(x=after[c], ax=axes[i, 1]).set_title(f'{c}: capped')
plt.tight_layout()
plt.show()

---
## Step 9 · Drop rare binary features
**Rule 6.** For a 0/1 column the sum counts the 1s. Fewer than 10 ones, or fewer than 10 zeros, and it cannot help.

> **Question to consider:** Would a flag that 3 recipes have help classify the other 20,000? What about one that every recipe has?

In [ ]:
# For a 0/1 column, which aggregate counts the 1s? A column is too rare
# with fewer than ten 1s, or fewer than ten 0s.
# one aggregation over every binary column, returned as a dictionary
sums = food3.select(*[F.___(F.col(c)).alias(c) for c in BINARY_COLUMNS]).head().asDict()
num_rows = food3.count()

too_rare_features = [c for c, v in sums.items() if v < ___ or v > (num_rows - ___)]
BINARY_COLUMNS = list(set(BINARY_COLUMNS) - set(too_rare_features))

print(len(too_rare_features), 'rare columns dropped;', len(BINARY_COLUMNS), 'binary columns left')
too_rare_features[:10]

---
## Step 10 · Create features from domain knowledge
**Rule 7.** Grams of fat mix up big dishes and fatty ones. The share of calories from fat separates them.

> **Question to consider:** Protein has 4 calories per gram and fat has 9. What does a fat_ratio of 0.6 mean?

In [ ]:
# Protein has 4 calories per gram, fat has 9.
food3 = (food3
         .withColumn('protein_ratio', F.col('protein') * ___ / F.col('calories'))
         .withColumn('fat_ratio', F.col('fat') * ___ / F.col('calories')))

# a recipe with zero calories gives a missing ratio
food3 = food3.fillna(0.0, subset=['protein_ratio', 'fat_ratio'])
CONTINUOUS_COLUMNS += ['protein_ratio', 'fat_ratio']

In [ ]:
# Check: do the new features differ between desserts and other dishes?
food3.groupBy('dessert').agg(F.round(F.avg('protein_ratio'), 3).alias('avg protein_ratio'),
                             F.round(F.avg('fat_ratio'), 3).alias('avg fat_ratio')).show()

---
## Step 11 · Check for redundant features
**Rule 8.** Features should correlate with the target, not with each other.

> **Question to consider:** Calories and fat correlate at about 0.92. If you had to keep one, which would you keep, and why?

In [ ]:
# the quick way, for a handful of columns: bring them to pandas
corr = food3.select(CONTINUOUS_COLUMNS).toPandas().corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation between continuous features')
plt.show()

In [ ]:
# The method every transformer uses to produce its output, and the
# Correlation function that computes the matrix.
# the PySpark way, which scales to many columns: one vector column, then Correlation.corr
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

no_nulls = food3.select(CONTINUOUS_COLUMNS).dropna()      # VectorAssembler rejects nulls
vector_food = VectorAssembler(inputCols=CONTINUOUS_COLUMNS,
                              outputCol='continuous_features').___(no_nulls)

matrix = Correlation.___(vector_food, 'continuous_features').head()[0].toArray()
pd.DataFrame(matrix, index=CONTINUOUS_COLUMNS, columns=CONTINUOUS_COLUMNS).round(2)

---
## Step 12 · Transformers and estimators
**Rule 10.** Steps that learn something from the data should be learned once and reused.

| | Transformer | Estimator |
|---|---|---|
| What it does | Changes a DataFrame by a fixed rule | Learns a rule from the data first |
| Method | `transform(df)` | `fit(df)` returns a model, then `model.transform(df)` |
| Learns from the data? | No | Yes: a mean, a minimum and maximum, ... |
| Examples today | `VectorAssembler` | `Imputer`, `MinMaxScaler` |

> **Question to consider:** `VectorAssembler` has no `fit()`. Why does it not need one?

---
## Step 13 · Fill continuous gaps with the Imputer
**Rule 3 and 10.** About a fifth of recipes have no nutrition values. The Imputer learns each column's mean, then fills.

> **Question to consider:** The mean or the median: which suits a skewed column like sodium better?

In [ ]:
# Fill with the average. An estimator first LEARNS from the data, then
# applies what it learned.
from pyspark.ml.feature import Imputer

OLD_COLS = ['rating', 'calories', 'protein', 'fat', 'sodium']
NEW_COLS = [c + '_i' for c in OLD_COLS]

imputer = Imputer(strategy='___', inputCols=OLD_COLS, outputCols=NEW_COLS)
imputer_model = imputer.___(food3)          # the learning happens here
food_imputed = imputer_model.___(food3)

CONTINUOUS_COLUMNS = list(set(CONTINUOUS_COLUMNS) - set(OLD_COLS)) + NEW_COLS

In [ ]:
# What did fit() learn? One mean per column, stored in the model
imputer_model.surrogateDF.show()

---
## Step 14 · Assemble, then scale to 0 to 1
**Rule 9.** Scalers work on one vector column, so assemble the features first.

> **Question to consider:** Which models care about scale? (Think about distances and gradients versus decision trees.)

In [ ]:
# Same two steps as the Imputer: learn the min and max, then apply them.
from pyspark.ml.feature import MinMaxScaler

CONTINUOUS_NB = [c for c in CONTINUOUS_COLUMNS if 'ratio' not in c]   # the ratios are already 0 to 1

assembled = VectorAssembler(inputCols=CONTINUOUS_NB,
                            outputCol='continuous').transform(food_imputed)

scaler = MinMaxScaler(inputCol='continuous', outputCol='continuous_scaled')
scaler_model = scaler.___(assembled)        # learns each feature's min and max
food_features = scaler_model.___(assembled)

food_features.select('continuous', 'continuous_scaled').show(3, False)

In [ ]:
# What did fit() learn? The minimum and maximum of each feature, in CONTINUOUS_NB order
pd.DataFrame({'feature': CONTINUOUS_NB,
              'min': scaler_model.originalMin.toArray(),
              'max': scaler_model.originalMax.toArray()})

---
## Step 15 · Learn from training data only
**Rule 10.** Above, we fitted on all the data to keep things short. A real model must split first, fit on the training set, and apply the same learned values to the test set.

> **Question to consider:** If the test set helps compute the means, why would the model look better than it really is?

In [ ]:
# Learn the means from the training data only, then apply them to the test data.
train, test = food3.randomSplit([0.8, 0.2], seed=42)

imputer_model = imputer.fit(___)          # means learned from training data only
train_imputed = imputer_model.transform(train)
test_imputed = imputer_model.transform(___)    # the same means applied to the test set

print('training rows:', train.count(), '  test rows:', test.count())
imputer_model.surrogateDF.show()

**Coming next, Chapter 13:** a `Pipeline` chains the assembler, imputer, scaler and
model into one estimator, so the whole preparation is fitted on the training data
once and reused everywhere.

---
## Wrap-up

**Exit ticket** (one sentence each):
1. Name one rule from today and the PySpark tool that applied it.
2. Why did we cap extreme values instead of deleting those recipes?
3. What is the difference between a transformer and an estimator?
4. Why must `fit()` see only the training data?

In [ ]:
spark.stop()